# Ground rules

Using Server "Galaktische Republik" for testing

Step 1: Data Preparation


In [330]:
import os.path
import pandas as pd

# Place the id of the dir you wanna use here
gr_id = '524734557220634624'

test_data_path = os.path.join('data', gr_id)
dev_config_path = os.path.join(test_data_path, 'config.json')
dev_event_log_path = os.path.join(test_data_path, 'event_log.csv')


In [331]:
df_voice_events = pd.read_csv(dev_event_log_path)

In [332]:
print(df_voice_events)

               member_id     member_name     timestamp            guild_id  \
0     176034509014171648    realfirehawk  1.770077e+09  524734557220634624   
1     608068617069789369  merilineth7638  1.770077e+09  524734557220634624   
2     608068617069789369  merilineth7638  1.770078e+09  524734557220634624   
3     176034509014171648    realfirehawk  1.770078e+09  524734557220634624   
4     372767039657345036     obihornchen  1.770144e+09  524734557220634624   
...                  ...             ...           ...                 ...   
3013  691704924693594125         hiplox.  1.786141e+09  524734557220634624   
3014  796837302096887879       thetime11  1.786143e+09  524734557220634624   
3015  474281490822463488        tobeyyyy  1.786143e+09  524734557220634624   
3016  796837302096887879       thetime11  1.786143e+09  524734557220634624   
3017  796837302096887879       thetime11  1.786143e+09  524734557220634624   

                    guild_name           channel_id        chan

## Data Preparation

We need to do some fun things

- Convert timestamps into pandas datetimes
- Set member names to every members last known name
- Set channel names to every channels last known name
- Handle orphan joins
- Save Server name once and drop it from the table

### Prepare a copy of the dataframe

In [333]:
df_voice_events_prepped = df_voice_events.copy()

### Convert the timestamps into pandas datetimes

In [334]:
df_voice_events_prepped['timestamp'] = pd.to_datetime(df_voice_events_prepped['timestamp'], unit='s')

### Save the server name plus id and then drop them from the table

In [335]:
# Getting the last row of the df so we get the last name the server was saved as
last_row = df_voice_events_prepped.sort_values(by=['timestamp'], ascending=True).tail(1)
guild_id = last_row['guild_id'].values[0]
guild_name = last_row['guild_name'].values[0]
print(guild_id)
print(guild_name)
df_voice_events_prepped.drop(columns=['guild_id', 'guild_name'], inplace=True)

524734557220634624
Die Galaktische Republik


### Set the members names to their last known name, identified by id

In [336]:
member_ids = df_voice_events_prepped['member_id'].unique()
print(f'There are {len(member_ids)} members in this guild')
print(f'Members by id: {df_voice_events_prepped['member_id'].nunique()}, by name: {df_voice_events_prepped['member_name'].nunique()}')
member_id_name_map = {}

for m_id in member_ids:
    df_member = df_voice_events_prepped[df_voice_events_prepped['member_id'] == m_id]
    last_row = df_member.sort_values(by=['timestamp'], ascending=True).tail(1)
    last_known_name = last_row['member_name'].values[0]
    member_id_name_map[m_id] = last_known_name

df_voice_events_prepped.drop(columns=['member_name'], inplace=True)
print(member_id_name_map)

There are 18 members in this guild
Members by id: 18, by name: 19
{np.int64(176034509014171648): 'realfirehawk', np.int64(608068617069789369): 'merilineth7638', np.int64(372767039657345036): 'obihornchen', np.int64(770367618921398290): 'sakawe', np.int64(675740751199600671): 'mrblanc.', np.int64(442768279388291072): 'olorin4', np.int64(286156392312733697): 'coldjack', np.int64(1007986223631048704): 'viv03619', np.int64(124567572397031425): 'schlamasl', np.int64(185812748192448512): '.luck3r.', np.int64(713465732704501790): 'tb.asm', np.int64(1253824766507225179): 'darkempress1234', np.int64(796837302096887879): 'thetime11', np.int64(474281490822463488): 'tobeyyyy', np.int64(691704924693594125): 'hiplox.', np.int64(1505237993948971008): 'mithrandir0137_65788', np.int64(1523070486236365007): 'lugginator', np.int64(1319315891355258990): 'blockbladig_19921'}


### Do the same for the channels

In [337]:
channel_ids = df_voice_events_prepped['channel_id'].unique()
print(f'There are {len(channel_ids)} used channels in this guild')
print(f'Channels by id: {df_voice_events_prepped['channel_id'].nunique()}, by name: {df_voice_events_prepped['channel_name'].nunique()}')
channel_id_name_map = {}
for c_id in channel_ids:
    df_channel = df_voice_events_prepped[df_voice_events_prepped['channel_id'] == c_id]
    last_row = df_channel.sort_values(by=['timestamp'], ascending=True).tail(1)
    last_known_name = last_row['channel_name'].values[0]
    channel_id_name_map[c_id] = last_known_name

df_voice_events_prepped.drop(columns=['channel_name'], inplace=True)

print(channel_id_name_map)

There are 9 used channels in this guild
Channels by id: 9, by name: 9
{np.int64(1370422030939328675): 'Anakins Gemächer', np.int64(1370422156898336769): "Cal Kestis' Gemächer", np.int64(1370422095854567454): 'Gemächer des Kanzlers', np.int64(1370446094009372672): 'Meditationskammer (bitte Ruhe)', np.int64(1480033896153485455): 'Ratsdiskussionen', np.int64(1394685224993951806): 'Studiersaal', np.int64(963198266600722532): 'Mos Eisley Cantina', np.int64(1370435474107138118): 'Jabbas Palast', np.int64(1370426153512472599): 'Der Senat'}


### Add a column that tracks if events were generated or not

In [338]:
df_voice_events_prepped.insert(loc=4, column='synthetic', value=False)

### Before Orphan fixing let us take a snapshot to see if all goes well

In [339]:
print(df_voice_events_prepped)

def verify_event_counts(df):
    counts = df.groupby(['member_id', 'event_type']).size().unstack(fill_value=0)
    print(counts)
    return (counts['join'] == counts['leave']).all()

print(verify_event_counts(df_voice_events_prepped))


               member_id                     timestamp           channel_id  \
0     176034509014171648 2026-02-03 00:07:14.684171677  1370422030939328675   
1     608068617069789369 2026-02-03 00:07:58.618698835  1370422030939328675   
2     608068617069789369 2026-02-03 00:19:05.147214890  1370422030939328675   
3     176034509014171648 2026-02-03 00:19:06.463059425  1370422030939328675   
4     372767039657345036 2026-02-03 18:32:40.073635817  1370422030939328675   
...                  ...                           ...                  ...   
3013  691704924693594125 2026-08-07 22:19:11.810528517   963198266600722532   
3014  796837302096887879 2026-08-07 22:46:34.598863840   963198266600722532   
3015  474281490822463488 2026-08-07 22:49:58.627666712   963198266600722532   
3016  796837302096887879 2026-08-07 22:55:23.410823107   963198266600722532   
3017  796837302096887879 2026-08-07 22:55:57.541040897   963198266600722532   

     event_type  synthetic  
0          join      F

### Handle orphan joins

In [344]:
print('df before checking for orphans:')
print(df_voice_events_prepped)
print('============================')

def insert_row(member_id, timestamp, channel_id, event_type):
    global df_voice_events_prepped

    if event_type == 'join':
        timestamp -= pd.Timedelta(seconds=1)
    else:
        timestamp += pd.Timedelta(seconds=1)

    new_row = pd.DataFrame([{
        'member_id': member_id,
        'timestamp': timestamp,
        'channel_id': channel_id,
        'event_type': event_type,
        'synthetic': True,
    }])

    df_voice_events_prepped = pd.concat([df_voice_events_prepped, new_row], ignore_index=True)

def correct_orphan_joins(df_member):
    # We expect every df to start with a join
    expected_type = 'join'
    df_member.reset_index(inplace=True)
    rows = 0
    for idx, row in df_member.iterrows():
        rows += 1
        actual_type = row['event_type']
        # If the types match we can swap the expected type and move on
        if actual_type == expected_type:
            if expected_type == 'join':
                expected_type = 'leave'
            else:
                expected_type = 'join'
        else:
            # If they dont we insert the missing event in our original df
            insert_row(row['member_id'], row['timestamp'], row['channel_id'], expected_type)

    # This essentially means we are missing a leave event at the end that is not being caught since the for loop simply terminates
    if rows % 2 == 1 and expected_type == 'leave':
        last_row = df_member.tail(1).iloc[0]
        insert_row(last_row['member_id'], last_row['timestamp'], last_row['channel_id'], expected_type)



# Copy the df to avoid any shenanigans where we get errors due to the df being looped over and changed at the same time
df_copy = df_voice_events_prepped.copy(deep=True)

for m_id in member_ids:

    df_member = df_copy[df_copy['member_id'] == m_id]
    correct_orphan_joins(df_member)

# ====== chatty from here =======

# Sort chronologically.
# For identical timestamps, leave comes before join since this must be a switch of channels.
df_voice_events_prepped['event_order'] = (
    df_voice_events_prepped['event_type'] == 'join'
).astype(int)

df_voice_events_prepped.sort_values(
    by=['timestamp', 'event_order'],
    inplace=True
)

df_voice_events_prepped.drop(
    columns=['event_order'],
    inplace=True
)

df_voice_events_prepped.reset_index(
    drop=True,
    inplace=True
)

# ======= chatty end =======

print('============================')
print('df after checking for orphans:')
print(df_voice_events_prepped)


df before checking for orphans:
               member_id                     timestamp           channel_id  \
0     176034509014171648 2026-02-03 00:07:14.684171677  1370422030939328675   
1     608068617069789369 2026-02-03 00:07:58.618698835  1370422030939328675   
2     608068617069789369 2026-02-03 00:19:05.147214890  1370422030939328675   
3     176034509014171648 2026-02-03 00:19:06.463059425  1370422030939328675   
4     372767039657345036 2026-02-03 18:32:40.073635817  1370422030939328675   
...                  ...                           ...                  ...   
3019  474281490822463488 2026-08-07 22:49:58.627666712   963198266600722532   
3020  474281490822463488 2026-08-07 22:49:59.627666712   963198266600722532   
3021  796837302096887879 2026-08-07 22:55:23.410823107   963198266600722532   
3022  796837302096887879 2026-08-07 22:55:57.541040897   963198266600722532   
3023  796837302096887879 2026-08-07 22:55:58.541040897   963198266600722532   

     event_type  sy

In [345]:

print(verify_event_counts(df_voice_events_prepped))


event_type           join  leave
member_id                       
124567572397031425    184    184
176034509014171648    274    274
185812748192448512     64     64
286156392312733697     14     14
372767039657345036     68     68
442768279388291072    195    195
474281490822463488     78     78
608068617069789369    195    195
675740751199600671     65     65
691704924693594125     54     54
713465732704501790     56     56
770367618921398290    138    138
796837302096887879     32     32
1007986223631048704    38     38
1253824766507225179     1      1
1319315891355258990     4      4
1505237993948971008    46     46
1523070486236365007     7      7
True


In [343]:
# Saving one members data fordebug reasons
df_test = df_voice_events_prepped[df_voice_events_prepped['member_id'] == 124567572397031425]
print(df_test)
pd.DataFrame.to_csv(df_test, 'test.csv', index=False)

               member_id                     timestamp           channel_id  \
34    124567572397031425 2026-02-08 23:13:17.145958185  1370422030939328675   
35    124567572397031425 2026-02-08 23:13:18.145958185  1370422030939328675   
46    124567572397031425 2026-02-09 20:00:52.325712919  1370422030939328675   
53    124567572397031425 2026-02-10 00:36:15.109276772  1370422030939328675   
61    124567572397031425 2026-02-10 21:04:18.368573904  1370422030939328675   
...                  ...                           ...                  ...   
2993  124567572397031425 2026-08-05 23:55:33.503370285  1370422030939328675   
2997  124567572397031425 2026-08-06 19:32:52.865567923  1370422030939328675   
2998  124567572397031425 2026-08-06 19:35:03.863398552  1370422030939328675   
2999  124567572397031425 2026-08-06 19:57:58.118973017  1370422030939328675   
3012  124567572397031425 2026-08-07 00:59:10.839706182  1370422030939328675   

     event_type  synthetic  
34         join       